# Notebook 03b — Class Imbalance Strategies

**Question:** Does SMOTE / undersampling actually help beyond `class_weight='balanced'`?

We take the top models from nb03a and test them across different resampling strategies
on the 4-class problem. The 3-class and binary results from nb03a serve as the comparison
point — reframing the problem vs resampling it.

**Inputs:** `artifacts/` from nb02, `results/ml_baselines.csv` from nb03a
**Outputs:** `results/ml_imbalance.csv`


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, time, os, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb
import lightgbm as lgb

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style("whitegrid")
SEED = 42; np.random.seed(SEED)
os.makedirs('results', exist_ok=True)
print("Imports OK")


## 0. Load Data & Baselines

In [ ]:
# Load data
X_train = joblib.load('../artifacts/X_train.pkl').astype(np.float32)
X_val   = joblib.load('../artifacts/X_val.pkl').astype(np.float32)
y_train = joblib.load('../artifacts/y_train.pkl')
y_val   = joblib.load('../artifacts/y_val.pkl')
le_target = joblib.load('../artifacts/label_encoder_target.pkl')
feature_names = joblib.load('../artifacts/feature_names.pkl')

X_4c = np.vstack([X_train, X_val])
y_4c = np.concatenate([y_train, y_val])

# Load baselines for comparison
baselines = pd.read_csv('results/ml_baselines.csv')
print(f'4-class data: {X_4c.shape}')
print(f'Distribution: {dict(zip(le_target.classes_, np.bincount(y_4c)))}')
print(f'\nBaseline scores (4-class, from nb03a):')
b4 = baselines[baselines['strategy'] == '4class'].sort_values('cv_f1_mean', ascending=False)
print(b4[['model', 'cv_f1_mean', 'cv_f1_std']].head(5).to_string(index=False))


In [ ]:
SCOREBOARD = []

def log_exp(model, strategy, cv_mean, cv_std, n_train, notes=""):
    SCOREBOARD.append(dict(
        phase="imbalance", model=model, strategy=strategy,
        n_features=len(feature_names),
        cv_f1_mean=round(cv_mean, 4), cv_f1_std=round(cv_std, 4),
        n_train=n_train, notes=notes
    ))
    print(f"  {model:15s} | {strategy:20s} | F1={cv_mean:.4f} ± {cv_std:.4f}")

def run_cv(model, X, y, n_folds=5):
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro", n_jobs=-1)
    return scores.mean(), scores.std()

def undersample_op(X, y, target=15000):
    op = le_target.transform(["operating"])[0]
    oi = np.where(y == op)[0]; oth = np.where(y != op)[0]
    rng = np.random.RandomState(SEED)
    keep = np.concatenate([rng.choice(oi, min(target, len(oi)), replace=False), oth])
    rng.shuffle(keep)
    return X[keep], y[keep]


## 1. SMOTE Strategies (on 4-class)

We test SMOTE and SMOTETomek inside imblearn Pipelines so resampling happens inside
each CV fold — no data leakage.


In [ ]:
print("=== SMOTE Strategies ===")

# SMOTE + XGBoost
pipe_smote_xgb = ImbPipeline([
    ("smote", SMOTE(random_state=SEED, k_neighbors=5)),
    ("model", xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric="mlogloss"))
])
m, s = run_cv(pipe_smote_xgb, X_4c, y_4c)
log_exp("XGBoost", "SMOTE", m, s, len(X_4c))

# SMOTE + LightGBM
pipe_smote_lgb = ImbPipeline([
    ("smote", SMOTE(random_state=SEED, k_neighbors=5)),
    ("model", lgb.LGBMClassifier(n_estimators=200, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1))
])
m, s = run_cv(pipe_smote_lgb, X_4c, y_4c)
log_exp("LightGBM", "SMOTE", m, s, len(X_4c))

# SMOTETomek + LightGBM
pipe_st_lgb = ImbPipeline([
    ("smotetomek", SMOTETomek(random_state=SEED)),
    ("model", lgb.LGBMClassifier(n_estimators=200, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1))
])
m, s = run_cv(pipe_st_lgb, X_4c, y_4c)
log_exp("LightGBM", "SMOTE+Tomek", m, s, len(X_4c))


## 2. Undersampling Strategies

Instead of oversampling minorities, we reduce the majority class (operating).
This removes the noisy, low-signal operating startups that overlap with closed.


In [ ]:
print("=== Undersampling Strategies ===")

for target in [15000, 10000, 6000]:
    Xu, yu = undersample_op(X_4c, y_4c, target)
    dist = dict(zip(le_target.classes_, np.bincount(yu)))
    print(f'\n  Undersample to {target}: {len(Xu)} rows — {dist}')

    m, s = run_cv(lgb.LGBMClassifier(n_estimators=200, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1), Xu, yu)
    log_exp("LightGBM", f"under_{target//1000}k", m, s, len(Xu))

    m, s = run_cv(xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric="mlogloss"), Xu, yu)
    log_exp("XGBoost", f"under_{target//1000}k", m, s, len(Xu))


## 3. Hybrid: Undersample + SMOTE

In [ ]:
print("=== Hybrid: Undersample 15k + SMOTE ===")

Xu15, yu15 = undersample_op(X_4c, y_4c, 15000)

hybrid = ImbPipeline([
    ("smote", SMOTE(random_state=SEED)),
    ("model", lgb.LGBMClassifier(n_estimators=200, class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1))
])
m, s = run_cv(hybrid, Xu15, yu15)
log_exp("LightGBM", "under_15k+SMOTE", m, s, len(Xu15))

hybrid_xgb = ImbPipeline([
    ("smote", SMOTE(random_state=SEED)),
    ("model", xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=6,
        random_state=SEED, n_jobs=-1, verbosity=0, eval_metric="mlogloss"))
])
m, s = run_cv(hybrid_xgb, Xu15, yu15)
log_exp("XGBoost", "under_15k+SMOTE", m, s, len(Xu15))


## 4. Results: Resampling vs Reframing

In [ ]:
sb = pd.DataFrame(SCOREBOARD)

# Add baselines and 3-class/binary from nb03a for comparison
baselines_compare = baselines[
    (baselines['strategy'].isin(['4class', '3class', 'binary'])) &
    (baselines['model'].isin(['XGBoost', 'LightGBM']))
].copy()
baselines_compare['phase'] = 'baseline_ref'

all_results = pd.concat([sb, baselines_compare], ignore_index=True)
all_results = all_results.sort_values('cv_f1_mean', ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
all_results['label'] = all_results['model'] + ' + ' + all_results['strategy']
all_results_plot = all_results.sort_values('cv_f1_mean', ascending=True)

palette = {
    '4class': '#888780', '3class': '#534AB7', 'binary': '#1D9E75',
    'SMOTE': '#D85A30', 'SMOTE+Tomek': '#D85A30',
    'under_15k': '#378ADD', 'under_10k': '#378ADD', 'under_6k': '#378ADD',
    'under_15k+SMOTE': '#E8A735',
}
colors = [palette.get(r['strategy'], '#888780') for _, r in all_results_plot.iterrows()]

bars = ax.barh(all_results_plot['label'], all_results_plot['cv_f1_mean'],
               color=colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, all_results_plot['cv_f1_mean']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('Macro F1')
ax.set_title('Resampling Strategies vs Problem Reframing')
ax.set_xlim(0, 0.85)
plt.tight_layout()
plt.savefig('results/nb03b_imbalance_comparison.png', bbox_inches='tight')
plt.show()

print('\nConclusion:')
print('  Reframing (3-class/binary) >> any resampling strategy on 4-class')


In [ ]:
sb.to_csv('results/ml_imbalance.csv', index=False)
print(f'Saved: results/ml_imbalance.csv ({len(sb)} experiments)')
print()
print(sb.sort_values('cv_f1_mean', ascending=False).to_string(index=False))
